# Day 11: Transformers, Hugging Face & LLM Basics

## Why Transformers Replaced RNN/LSTM
Self-attention lets a model weigh every word against every other word in parallel, instead of processing text one step at a time like an RNN/LSTM (Day 71 of the advanced track) - this is what made today's LLMs possible.

In [ ]:
# pip install transformers torch

## Using a Pretrained Pipeline
Hugging Face's pipeline() wraps tokenizer + model + post-processing so you get state-of-the-art results in one line.

In [ ]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis")
result = classifier("This course finally made deep learning make sense!")
print(result)

## Text Generation with a Small LLM
The same pipeline pattern works for generation - swap the task and the model checkpoint.

In [ ]:
from transformers import pipeline

generator = pipeline("text-generation", model="gpt2")
output = generator("Data science in 2026 will be about", max_length=30, num_return_sequences=1)
print(output[0]["generated_text"])

## Fine-Tuning Concept
Rather than training from scratch, you continue training a pretrained model on your own labeled data - far less data and compute needed than Day 64-70's from-scratch networks.

In [ ]:
# conceptual sketch (needs a labeled dataset + GPU to actually run):
# from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments
#
# model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)
# tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
#
# training_args = TrainingArguments(output_dir="./results", num_train_epochs=3, per_device_train_batch_size=16)
# trainer = Trainer(model=model, args=training_args, train_dataset=train_ds, eval_dataset=eval_ds)
# trainer.train()

## Calling a Hosted LLM API
Many production systems don't host a model at all - they call an LLM API and engineer the prompt to get reliable output.

In [ ]:
import requests

def ask_llm(prompt, api_url, api_key):
    response = requests.post(
        api_url,
        headers={"Authorization": f"Bearer {api_key}"},
        json={"prompt": prompt, "max_tokens": 100},
    )
    return response.json()

# ask_llm("Summarize this dataset's key trend in one sentence: ...", api_url="...", api_key="...")

# Prompt engineering basics:
# - Be specific about format: "Answer in exactly 3 bullet points."
# - Give an example of the output you want (few-shot).
# - Ask it to think step by step for reasoning-heavy tasks.

## RAG — Retrieval-Augmented Generation (Concept)
Instead of relying only on what an LLM memorized, RAG retrieves relevant chunks of your own documents (via embeddings + similarity search) and feeds them into the prompt - this is how most real company chatbots ground answers in their own data.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# toy example: "embeddings" as simple word-overlap vectors, standing in for a real embedding model
documents = [
    "Random Forest averages many decision trees to reduce variance.",
    "Gradient Boosting builds trees sequentially to fix previous errors.",
    "PCA reduces dimensionality by finding directions of maximum variance.",
]
query = "How does boosting work?"

# in a real RAG pipeline: embeddings = OpenAIEmbeddings() / SentenceTransformer(...)
vocab = sorted(set(" ".join(documents + [query]).lower().split()))
def to_vector(text):
    words = text.lower().split()
    return np.array([words.count(w) for w in vocab])

doc_vectors = np.array([to_vector(d) for d in documents])
query_vector = to_vector(query).reshape(1, -1)

scores = cosine_similarity(query_vector, doc_vectors)[0]
best_doc = documents[np.argmax(scores)]
print("Most relevant context to feed the LLM:", best_doc)